In [13]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torch.utils.data import DataLoader, ConcatDataset
from avalanche.benchmarks.classic import SplitMNIST

from skill_memory import (
    SkillMemoryConfig,
    SkillClassifierBank,
    SkillMemoryStrategy,
    evaluate_seen_experiences,
    compute_cl_metrics,
)

config = SkillMemoryConfig(
    n_skills=10,
    input_dim=784,
    num_classes=10,
    batch_size=64,
    epochs_per_experience=1,
    learning_rate=0.01,
    probe_batch_size=10,
    probe_batches=5,       # was implicitly 1 batch of 10; now 5 batches (~50 samples)
    forgetting_margin=0.05,
    replay_old_during_reuse=False,  # flip to True to A/B against replay
    replay_batches_per_epoch=1,
    verbose=True,
    seed=0,                # set for reproducible probe sampling across runs
    device=str(device),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [14]:
benchmark = SplitMNIST(
    n_experiences=10,
    seed=SEED,
)

train_stream = benchmark.train_stream
test_stream = benchmark.test_stream

print("Number of experiences:", len(train_stream))

for i, exp in enumerate(train_stream):
    print(i, sorted(exp.classes_in_this_experience), len(exp.dataset))


Number of experiences: 10
0 [5] 5421
1 [6] 5918
2 [1] 6742
3 [2] 5958
4 [0] 5923
5 [8] 5851
6 [9] 5949
7 [3] 6131
8 [7] 6265
9 [4] 5842


In [15]:
class LinearMNIST(nn.Module):
    def __init__(self, n_skills=10):
        super().__init__()

        self.skill_classifiers = nn.ModuleList(
            [nn.Linear(784, 10) for _ in range(n_skills)]
        )

        # Skill slots that have actually been learned.
        self.learned_skills = set()

        # Skill used by forward().
        self.active_skill = None

    def allocate_skill(self):
        for skill_index in range(len(self.skill_classifiers)):
            if skill_index not in self.learned_skills:
                self.learned_skills.add(skill_index)
                return skill_index

        raise RuntimeError("No free skill slots available.")

    def set_active_skill(self, skill_index):
        self.active_skill = skill_index

    def forward(self, x):
        x = x.flatten(start_dim=1)

        if self.active_skill is None:
            raise RuntimeError("No active skill selected.")

        return self.skill_classifiers[self.active_skill](x)


## Evaluation

After each training experience, evaluate the same model on every test experience.

This gives us an accuracy matrix:

`accuracy_history[t][j]`

where `t` is the current training experience and `j` is the test experience.


In [16]:
model = SkillClassifierBank(config).to(config.device)

optimizer = torch.optim.SGD(model.parameters(), lr=config.learning_rate)

criterion = torch.nn.CrossEntropyLoss()

strategy = SkillMemoryStrategy(
    model, 
    optimizer, 
    criterion, 
    config
)

accuracy_history = []

for t, train_exp in enumerate(train_stream):
    strategy.train_experience(train_exp)

    current_accuracies = evaluate_seen_experiences(strategy, test_stream, t)
    accuracy_history.append(current_accuracies)

    print(
        f"Experience {t}: "
        f"trained on {sorted(train_exp.classes_in_this_experience)}, "
        f"mean seen accuracy = {np.mean(current_accuracies):.3f}"
    )

No existing skills -> allocated skill 0
Evaluation using skill 0: loss=0.0092, score=0.991, accuracy=1.000
Experience 0: trained on [5], mean seen accuracy = 1.000

Imagination:
  skill 0: old_score=0.992, old_acc=1.000, new_score=0.000, new_acc=0.000

No compatible existing skill -> allocated skill 1
Evaluation using skill 0: loss=0.0092, score=0.991, accuracy=1.000
Evaluation using skill 1: loss=0.0107, score=0.989, accuracy=1.000
Experience 1: trained on [6], mean seen accuracy = 1.000

Imagination:
  skill 0: old_score=0.057, old_acc=0.600, new_score=0.002, new_acc=0.000
  skill 1: old_score=0.045, old_acc=0.400, new_score=0.005, new_acc=0.000
Score candidates: []
Accuracy candidates: []
Compatible candidates: []

No compatible existing skill -> allocated skill 2
Evaluation using skill 0: loss=0.0092, score=0.991, accuracy=1.000
Evaluation using skill 1: loss=0.0107, score=0.989, accuracy=1.000
Evaluation using skill 2: loss=0.0063, score=0.994, accuracy=1.000
Experience 2: trained

In [17]:
accuracy_curve, forgetting_curve = compute_cl_metrics(accuracy_history)

print("Accuracy:", np.round(accuracy_curve, 3))
print("Forgetting:", np.round(forgetting_curve, 3))

Accuracy: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Forgetting: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
